In [ ]:
import pandas as pd

from pipeline import *

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
raw_to_processed("test")

In [ ]:
years = ["2023", "2024", "2025"]
for year in years:
    raw_to_processed(year)

In [ ]:
weight_by_parents = True
year = "2024"
raw_to_processed(year)
df = load_flattened(year)
df, average_results = compute_averages_and_weighted_totals(df, weight_by_parents)
for key in ["Overall Average", "Grammar Average", "Middle Average", "High Average"]:
    print(f"{key}: {average_results[key]:.2f}")

In [ ]:
plot_question_trends_across_years(years, savefig=True, save_csv=True)

In [ ]:
weight_by_parents = False
year = "2025"
raw_to_processed(year)
df = load_flattened(year)
rolled_up_data = calculate_question_totals(df, weight_by_parents)
plts = plot_individual_question_stacked_bars(year, rolled_up_data, savefig=True, save_csv=True)

In [ ]:
plot_sequence("2025", "Total", rolled_up_data, savefig=True, save_csv=True)

In [ ]:
weight_by_parents = False
year = "2024"
raw_to_processed(year)
df = load_flattened(year)
rolled_up_data = calculate_question_totals(df, weight_by_parents)

# First, pivot to get counts per question/response like the original function.
response_counts = rolled_up_data.groupby(["Question", "Response"])["N_total"].sum().unstack(fill_value=0)

# Initialize a dictionary to accumulate counts for each level (position in the response list)
level_counts = {}

# Loop through each question that we care about, excluding free-response ones.
for question in config.questions_for_each_school_level:
    if question in config.has_free_response:
        continue
    
    # Get the ordered list of responses for this question.
    responses = config.question_responses.get(question, [])
    
    # For each response option (by its index, which determines its level)
    for idx, response_name in enumerate(responses):
        # Use the pivoted DataFrame to get the count. If the response is missing for a question, default to 0.
        count = response_counts.loc[question, response_name] if response_name in response_counts.columns else 0
        level_counts[idx] = level_counts.get(idx, 0) + count

# Sum the counts across all levels to get the total
total = sum(level_counts.values())

# Compute proportions for each level.
# (If total==0, all levels will have proportion 0 to avoid division by zero.)
level_proportions = {idx: (count / total if total else 0) for idx, count in level_counts.items()}

print(level_counts)
print(level_proportions)

In [ ]:
weight_by_parents = True
rolled_up_data = calculate_question_totals(df, weight_by_parents)
rolled_up_data.head()

In [ ]:
rolled_up_data.groupby(["Question", "Response"])["N_total"].sum().unstack(fill_value=0) 